# vzviz: MUR SST Dashboard

A simple example showing how to visualize a remote MUR SST NetCDF file with `vzviz`.

Run with [juv](https://github.com/manzt/juv):
```bash
juv run examples/mur_sst_dashboard.ipynb
```

In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "earthaccess",
#     "virtualizarr[hdf] @ git+https://github.com/zarr-developers/VirtualiZarr",
#     "vzviz @ git+https://github.com/virtual-zarr/vzviz",
#     "obspec-utils @ git+https://github.com/virtual-zarr/obspec-utils@store-loop",
#     "aiohttp",
#     "pandas",
#     "jupyterlab",
#     "holoviews",
#     "panel",
#     "bokeh",
# ]
# ///

## Setup

Authenticate with NASA Earthdata and create a ManifestStore from a remote MUR SST file.

In [ ]:
from urllib.parse import urlparse

import earthaccess
import virtualizarr as vz
import vzviz
import holoviews as hv
import panel as pn

from obspec_utils.registry import ObjectStoreRegistry
from obspec_utils.stores import AiohttpStore

hv.extension("bokeh")
pn.extension()

In [ ]:
# Authenticate with NASA Earthdata
earthaccess.login()

In [ ]:
# Search for MUR SST data
results = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", count=1, temporal=("2002-06-01", "2002-06-01")
)

# Get the HTTPS URL
https_links = earthaccess.results.DataGranule.data_links(results[0], access="external")
https_url = https_links[0]
print(f"URL: {https_url}")

In [ ]:
# Parse URL and get auth token
parsed = urlparse(https_url)
base_url = f"{parsed.scheme}://{parsed.netloc}"
token = earthaccess.get_edl_token()["access_token"]

# Create store with authentication
store = AiohttpStore(
    base_url,
    headers={"Authorization": f"Bearer {token}"},
)
registry = ObjectStoreRegistry({base_url: store})

In [ ]:
# Create ManifestStore by parsing the NetCDF file
parser = vz.parsers.HDFParser()
manifest_store = parser(https_url, registry=registry)
print("ManifestStore created!")

# Show variables overview
overview = vzviz.variables_overview(manifest_store)
overview[["variable", "shape", "chunks", "dtype", "total_chunks", "total_bytes_human"]]

## Query Simulation

Simulate a data access pattern and analyze performance metrics.

In [ ]:
# Simulate reading a 1000x1000 spatial subset of analysed_sst
metrics = vzviz.simulate_query(
    manifest_store,
    "analysed_sst",
    {1: slice(0, 1000), 2: slice(0, 1000)},  # lat/lon subset
)

print("Performance Metrics:")
print(f"  Array Elements Requested: {metrics.requested_cells:,}")
print(f"  Array Elements Read:      {metrics.cells_read:,}")
print(f"  Read Amplification:       {metrics.read_amplification:.2f}x")
print(f"  Read Efficiency:          {metrics.read_efficiency:.1f}%")
print(f"  Chunks Touched:           {metrics.chunks_touched} / {metrics.total_chunks}")
print(f"  Range Reads:              {metrics.range_reads}")
print(f"  Coalescing Factor:        {metrics.coalescing_factor:.2f}x")
print(f"  Storage Alignment:        {metrics.storage_alignment:.2f}")

## Interactive Dashboard

Launch the interactive dashboard to explore the chunk manifest.

**Features:**
- **Variables table**: Select rows to filter chunks in ByteMap. Select one variable to view its ChunkMap.
- **ByteMap**: Shows chunk byte ranges within files, colored by variable.
- **ChunkMap**: Box-select a region to see performance metrics and highlight chunks in ByteMap.
- **Selection panel**: Shows selected chunks count, total size, and performance metrics.

In [ ]:
dashboard = vzviz.manifest_dashboard(manifest_store, variable="analysed_sst")
dashboard

### Serve in Browser

To serve the dashboard in a separate browser window:

In [ ]:
# Uncomment to serve in browser:
# dashboard.show()